# BHaH Project Anatomy

Read a standalone BHaH project generated from the Cartesian wave equation, then
match the generated files to their roles.

Navigation:
[Index](../index.ipynb) |
Previous: [Multicoordinate Wave Project][prev] |
Next: [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)

[prev]: ../3-wave_equation/wave_equation_multicoordinates.ipynb

## Learning Goals

- Read a generated BHaH project as a set of roles, not as a pile of files.
- Connect runtime settings, C headers, right-hand-side code, and diagnostics.
- Identify which files a beginner should inspect first.

## Words for This Notebook

- **BHaH:** BlackHoles@Home, the standalone C project style used by NRPy.
- **Prototype header:** a file listing C functions so source files can call them.
- **Source file:** a C file containing instructions to compile.
- **Diagnostic source:** C code that writes output used to check a run.
- **Installed package:** a Python package imported from `site-packages`.

Use the code cells actively: first predict what should happen, then run the cell.
Afterward, explain the output in plain language.

## Table of Contents

- [Generated-File Roles](#Generated-File-Roles)
- [Step 1: Import Project Inspection Helpers](#Step-1:-Import-Project-Inspection-Helpers)
- [Step 2: Create an Anatomy Workspace](#Step-2:-Create-an-Anatomy-Workspace)
- [Step 3: Generate a Fresh Anatomy Project](#Step-3:-Generate-a-Fresh-Anatomy-Project)
- [Step 4: Catalog the Generated Files](#Step-4:-Catalog-the-Generated-Files)
- [Validation Check](#Validation-Check)
- [Learning Check](#Learning-Check)
- [Step 5: Read the Parameter File](#Step-5:-Read-the-Parameter-File)
- [Step 6: Read the Prototype Header](#Step-6:-Read-the-Prototype-Header)
- [Step 7: Read Default-Setting Sources](#Step-7:-Read-Default-Setting-Sources)
- [Step 8: Read the Right-Hand-Side Source](#Step-8:-Read-the-Right-Hand-Side-Source)
- [Step 9: Read the Diagnostics Source](#Step-9:-Read-the-Diagnostics-Source)

## Generated-File Roles

| File | Role | What to inspect |
| --- | --- | --- |
| `Makefile` | Build recipe | Compiler target and source list |
| `wave_equation_cartesian.par` | Runtime settings | Editable parameter values |
| `BHaH_function_prototypes.h` | Function declarations | Callable C functions |
| `commondata_struct_set_to_default.c` | Defaults | Shared runtime values |
| `params_struct_set_to_default.c` | Defaults | Grid and equation parameters |
| `rhs_eval.c` | Wave update | Grid loop and RHS assignments |
| `diagnostics.c` | Error output | Error calculation and file write |

## Step 1: Import Project Inspection Helpers

These standard-library tools run commands, manage temporary project directories,
clean command output, and reject local NRPy source-tree imports.

If you are new to Python, skim this helper cell on a first pass. Its job is to
run terminal commands, shorten command output, and stop with a clear message if
a required command is missing.

In [1]:
from pathlib import Path
import os
import re
import subprocess
import sys
import tempfile


def local_nrpy_source_roots():
    roots = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "nrpy"
        if (candidate / "nrpy" / "__init__.py").is_file():
            roots.append(candidate.resolve())
    return roots


def require_installed_nrpy(import_path):
    resolved = Path(import_path).resolve()
    for source_root in local_nrpy_source_roots():
        if resolved.is_relative_to(source_root):
            raise RuntimeError(f"NRPy loaded from local source tree: {resolved}")
    installed_markers = {"site-packages", "dist-packages"}
    if not installed_markers.intersection(resolved.parts):
        raise RuntimeError(f"NRPy did not load from an installed package: {resolved}")
    return resolved


def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")


def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(
            args,
            cwd=cwd,
            env=os.environ.copy(),
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)

## Step 2: Create an Anatomy Workspace

The workspace keeps generated files separate from the tutorial source tree.

In [2]:
PROJECT_NAME = "wave_equation_cartesian"
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_cartesian_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME

print("workspace:", WORKSPACE)
print("project path:", PROJECT_DIR)

workspace: /work/5-infrastructures/nrpy_tutorial_cartesian_8ge2x_qn
project path: /work/5-infrastructures/nrpy_tutorial_cartesian_8ge2x_qn/project/wave_equation_cartesian


## Step 3: Generate a Fresh Anatomy Project

This command invokes the same module a learner can run from a terminal. The cell
then verifies that the project directory exists.

In [3]:
check_command = [
    sys.executable,
    "-c",
    "import nrpy, pathlib; print(pathlib.Path(nrpy.__file__).resolve())",
]
nrpy_import_path = Path(run_command(check_command, WORKSPACE, 30).strip())
nrpy_import_path = require_installed_nrpy(nrpy_import_path)
print("subprocess NRPy installed package:", nrpy_import_path)

command = [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"]
print("generator command: python -m nrpy.examples.wave_equation_cartesian")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)
print("project generated:", PROJECT_DIR.relative_to(WORKSPACE))

subprocess NRPy installed package: /virt/lib/python3.12/site-packages/nrpy/__init__.py
generator command: python -m nrpy.examples.wave_equation_cartesian


Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par
project generated: project/wave_equation_cartesian


## Step 4: Catalog the Generated Files

The inventory shows which files the generator produced before we inspect the
lesson-critical ones.

In [4]:
print("top-level project entries:")
for path in sorted(PROJECT_DIR.iterdir()):
    print(path.name + ("/" if path.is_dir() else ""))

top-level project entries:
BHaH_defines.h
BHaH_function_prototypes.h
Makefile
MoL/
apply_bcs.c
cmdline_input_and_parfile_parser.c
commondata_struct_set_to_default.c
diagnostics.c
exact_solution_single_Cartesian_point.c
griddata_free.c
initial_data.c
intrinsics/
main.c
numerical_grids_and_timestep.c
params_struct_set_to_default.c
progress_indicator.c
rhs_eval.c
set_CodeParameters-nopointer.h
set_CodeParameters-simd.h
set_CodeParameters.h
wave_equation_cartesian.par


## Validation Check

Before reading individual files, verify that the generated project contains the
artifacts this notebook teaches.

In [5]:
required_files = [
    "Makefile",
    "BHaH_function_prototypes.h",
    "commondata_struct_set_to_default.c",
    "params_struct_set_to_default.c",
    "rhs_eval.c",
    "diagnostics.c",
    "wave_equation_cartesian.par",
]
missing = [
    relative_path
    for relative_path in required_files
    if not (PROJECT_DIR / relative_path).is_file()
]
if missing:
    raise FileNotFoundError("Missing generated files: " + ", ".join(missing))
print("validation passed: required BHaH files are present")

validation passed: required BHaH files are present


## Learning Check

Before reading the generated files below, predict which file should contain
runtime settings and which should contain the wave update formula.

## Step 5: Read the Parameter File

This is the runtime file a learner edits before running the executable.

In [6]:
parameter_file_text = (PROJECT_DIR / "wave_equation_cartesian.par").read_text(
    encoding="utf-8", errors="replace"
)
print(parameter_file_text)

#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_output_every = 0.2  # (REAL)
###########################
###########################
### Module: nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData
sigma = 3.0                     # (REAL)
wavespeed = 1.0                 # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.MoLtimestepping.register_all
CFL_FACTOR = 0.5                # (REAL)
t_final = 8.0                   # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.diagnostics.progress_indicator
output_progress_every = 1       # (int)



## Step 6: Read the Prototype Header

The header shows the callable generated functions.

In [7]:
prototype_header_text = (PROJECT_DIR / "BHaH_function_prototypes.h").read_text(
    encoding="utf-8", errors="replace"
)
print(prototype_header_text)

void apply_bcs(const commondata_struct *restrict commondata, const params_struct *restrict params, REAL *restrict gfs);
void cmdline_input_and_parfile_parser(commondata_struct *restrict commondata, int argc, const char *argv[]);
void commondata_struct_set_to_default(commondata_struct *restrict commondata);
void diagnostics(commondata_struct *restrict commondata, griddata_struct *restrict griddata);
void exact_solution_single_Cartesian_point(const commondata_struct *restrict commondata, const params_struct *restrict params, const REAL xCart0,
                                           const REAL xCart1, const REAL xCart2, REAL *restrict exact_soln_UUGF, REAL *restrict exact_soln_VVGF);
void griddata_free(commondata_struct *restrict commondata, griddata_struct *restrict griddata,
                   const bool free_non_y_n_gfs_and_core_griddata_pointers);
void initial_data(const commondata_struct *restrict commondata, griddata_struct *restrict griddata);
int main(int argc, const char *arg

## Step 7: Read Default-Setting Sources

These files connect parameter names in the `.par` file to actual C assignments.
As you read, look for default values and the structure fields being filled.

In [8]:
for relative_path in [
    "commondata_struct_set_to_default.c",
    "params_struct_set_to_default.c",
]:
    default_source_text = (PROJECT_DIR / relative_path).read_text(
        encoding="utf-8", errors="replace"
    )
    print(f"--- {relative_path} ---")
    print(default_source_text)

--- commondata_struct_set_to_default.c ---
#include "BHaH_defines.h"

/**
 * Set commondata_struct to default values specified within NRPy.
 */
void commondata_struct_set_to_default(commondata_struct *restrict commondata) {
  memset(commondata, 0, sizeof(*commondata));
  // Set commondata_struct variables to default
  commondata->CFL_FACTOR = 0.5;               // nrpy.infrastructures.BHaH.MoLtimestepping.register_all::CFL_FACTOR
  commondata->NUMGRIDS = 1;                   // nrpy.grid::NUMGRIDS
  commondata->convergence_factor = 1.0;       // __main__::convergence_factor
  commondata->diagnostics_output_every = 0.2; // __main__::diagnostics_output_every
  commondata->output_progress_every = 1;      // nrpy.infrastructures.BHaH.diagnostics.progress_indicator::output_progress_every
  commondata->sigma = 3.0;                    // nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData::sigma
  commondata->t_final = 8.0;                  // nrpy.infrastructures.BHaH.MoLtimestep

## Step 8: Read the Right-Hand-Side Source

The full `rhs_eval.c` file contains the wave-equation update routine. Look for
the grid loop, grid reads, finite-difference expressions, and assignments to the
right-hand-side fields.

In [9]:
rhs_source_text = (PROJECT_DIR / "rhs_eval.c").read_text(
    encoding="utf-8", errors="replace"
)
print(rhs_source_text)

#include "BHaH_defines.h"
#include "intrinsics/simd_intrinsics.h"

/**
 * Set RHSs for wave equation.
 */
void rhs_eval(const commondata_struct *restrict commondata, const params_struct *restrict params, const REAL *restrict in_gfs,
              REAL *restrict rhs_gfs) {
#include "set_CodeParameters-simd.h"
#pragma omp parallel for
  for (int i2 = NGHOSTS; i2 < Nxx_plus_2NGHOSTS2 - NGHOSTS; i2++) {
    for (int i1 = NGHOSTS; i1 < Nxx_plus_2NGHOSTS1 - NGHOSTS; i1++) {
      for (int i0 = NGHOSTS; i0 < Nxx_plus_2NGHOSTS0 - NGHOSTS; i0 += SIMD_WIDTH) {

        /*
         * NRPy-Generated GF Access/FD Code, Step 1 of 2:
         * Read gridfunction(s) from main memory and compute FD stencils as needed.
         */
        const REAL_SIMD_ARRAY uu_i2m2 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1, i2 - 2)]);
        const REAL_SIMD_ARRAY uu_i2m1 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1, i2 - 1)]);
        const REAL_SIMD_ARRAY uu_i1m2 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1 - 2, i2)]);
        const RE

## Step 9: Read the Diagnostics Source

Diagnostics turn field data into numbers that can be checked after a run. Look
for the output filename, the error calculation, and the line that writes data.

In [10]:
diagnostics_source_text = (PROJECT_DIR / "diagnostics.c").read_text(
    encoding="utf-8", errors="replace"
)
print(diagnostics_source_text)

#include "BHaH_defines.h"
#include "BHaH_function_prototypes.h"

/**
 * Diagnostics.
 */
void diagnostics(commondata_struct *restrict commondata, griddata_struct *restrict griddata) {
  const REAL currtime = commondata->time, currdt = commondata->dt, outevery = commondata->diagnostics_output_every;
  // Explanation of the if() below:
  // Step 1: round(currtime / outevery) gives the nearest integer n to the ratio currtime/outevery.
  // Step 2: Multiplying by outevery yields the nearest output time t_out = n * outevery.
  // Step 3: If fabs(t_out - currtime) < 0.5 * currdt, then currtime is as close to t_out as possible.
  if (fabs(round(currtime / outevery) * outevery - currtime) < 0.5 * currdt) {
    for (int grid = 0; grid < commondata->NUMGRIDS; grid++) {
      // Unpack griddata struct:
      const REAL *restrict y_n_gfs = griddata[grid].gridfuncs.y_n_gfs;
      REAL *restrict xx[3];
      for (int ww = 0; ww < 3; ww++)
        xx[ww] = griddata[grid].xx[ww];
      const params_st

The inspected files show how a BHaH project connects runtime settings, C
function declarations, generated C functions, and diagnostics.

## Continue to ETLegacy Infrastructure

- [C Function Registration](../1-intro/c_function.ipynb)
- [Start-to-Finish Cartesian Wave Project][cart]
- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)

[cart]: ../3-wave_equation/start_to_finish_wave_cartesian.ipynb